In [1]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\DELL\Documents\zepto-capstone-project\.venv\Scripts\python.exe
3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]


# Module 1 - Zepto Data Pipeline

## Objective

Build an end-to-end data pipeline including:

- Web scraping
- Data cleaning
- Dataset creation
- SQL database storage
- SQL validation
- Pandas analysis

Import Libraries

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

Verify Libraries

In [4]:
print("Requests:", requests.__version__)
print("Pandas:", pd.__version__)
print("BeautifulSoup imported successfully")

Requests: 2.34.2
Pandas: 3.0.6
BeautifulSoup imported successfully


Create Headers

In [5]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9"
}

print(headers)

{'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36', 'Accept-Language': 'en-US,en;q=0.9'}


Test Internet Connection

In [6]:
url = "https://books.toscrape.com/"

response = requests.get(url, headers=headers)

print("Status Code:", response.status_code)

Status Code: 200


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

print("Environment is ready!")

Environment is ready!


In [2]:
import requests

url = "https://books.toscrape.com/"

response = requests.get(url)

print("Status Code:", response.status_code)

Status Code: 200


BeautifulSoup on the first page

In [3]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

books = soup.select("article.product_pod")

print("Books found on page:", len(books))

Books found on page: 20


test category 

In [4]:
# Get the URL of the first book
book_link = books[0].h3.a["href"]

print("Book link:", book_link)

Book link: catalogue/a-light-in-the-attic_1000/index.html


Get the category

In [5]:
from urllib.parse import urljoin

# Convert relative book link into a complete URL
book_url = urljoin(url, book_link)

# Open the book detail page
book_response = requests.get(book_url)

# Parse the detail page
book_soup = BeautifulSoup(book_response.text, "html.parser")

# Get category from the breadcrumb
breadcrumb = book_soup.select("ul.breadcrumb li a")
category = breadcrumb[-1].get_text(strip=True)

print("Book URL:", book_url)
print("Category:", category)

Book URL: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
Category: Poetry


Combine all 5 fields for ONE book

In [7]:
# Select the first book again
book = books[0]

# Extract all 5 required fields
title = book.h3.a["title"]
price = book.select_one("p.price_color").text.strip()
rating = book.select_one("p.star-rating")["class"][1]
availability = book.select_one("p.availability").get_text(strip=True)

print("Title:", title)
print("Price:", price)
print("Rating:", rating)
print("Availability:", availability)
print("Category:", category)

Title: A Light in the Attic
Price: Â£51.77
Rating: Three
Availability: In stock
Category: Poetry


Build the automated 60+ book collection

In [44]:
import time
from urllib.parse import urljoin

all_books = []

# Collect books from the first 5 pages = up to 100 books
for page_number in range(1, 6):

    page_url = f"https://books.toscrape.com/catalogue/page-{page_number}.html"

    page_response = requests.get(page_url)
    page_soup = BeautifulSoup(page_response.text, "html.parser")

    books_on_page = page_soup.select("article.product_pod")

    print(f"Page {page_number}: {len(books_on_page)} books found")

    for book in books_on_page:

        # Basic fields from listing page
        title = book.h3.a["title"]
        price = book.select_one("p.price_color").get_text(strip=True)
        rating = book.select_one("p.star-rating")["class"][1]
        availability = book.select_one("p.availability").get_text(strip=True)

        # Get individual book URL
        book_link = book.h3.a["href"]
        book_url = urljoin(page_url, book_link)

        # Open detail page
        book_response = requests.get(book_url)
        book_soup = BeautifulSoup(book_response.text, "html.parser")

        # Extract category
        breadcrumb = book_soup.select("ul.breadcrumb li a")
        category = breadcrumb[-1].get_text(strip=True)

        # Store all 5 required fields
        all_books.append({
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability,
            "category": category
        })

        time.sleep(0.1)

print("\nTotal books collected:", len(all_books))

Page 1: 20 books found
Page 2: 20 books found
Page 3: 20 books found
Page 4: 20 books found
Page 5: 20 books found

Total books collected: 100


Convert the 80 records into a Pandas DataFrame

In [45]:
df = pd.DataFrame(all_books)

print("Rows:", len(df))
print("Columns:", list(df.columns))

df.head()

Rows: 100
Columns: ['title', 'price', 'rating', 'availability', 'category']


,title,price,rating,availability,category
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction
2,Soumission,Â£50.10,One,In stock,Fiction
3,Sharp Objects,Â£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History


Verify categories across all 80 books

In [10]:
print("Number of unique categories:", df["category"].nunique())

print("\nCategories:")
print(df["category"].unique())
print("\nBooks per category:")
print(df["category"].value_counts())

Number of unique categories: 28

Categories:
<StringArray>
[            'Poetry', 'Historical Fiction',            'Fiction',
            'Mystery',            'History',        'Young Adult',
           'Business',            'Default',     'Sequential Art',
              'Music',    'Science Fiction',           'Politics',
             'Travel',           'Thriller',     'Food and Drink',
            'Romance',          'Childrens',         'Nonfiction',
                'Art',       'Spirituality',         'Philosophy',
          'New Adult',       'Contemporary',            'Fantasy',
      'Add a comment',            'Science',             'Health',
             'Horror']
Length: 28, dtype: str

Books per category:
category
Nonfiction            11
Default                9
Poetry                 7
Sequential Art         6
Fiction                5
Mystery                3
History                3
Young Adult            3
Music                  3
Thriller               3
Childrens   

Data Cleaning

In [46]:
df_clean = df.copy()

In [51]:
df_clean = df.copy()

print("Raw rows:", len(df))
print("Clean rows:", len(df_clean))

Raw rows: 100
Clean rows: 100


Inspect the data types

In [13]:
print(df_clean.dtypes)

title           str
price           str
rating          str
availability    str
category        str
dtype: object


Clean the Price column

In [14]:
print(df_clean["price"].head(10).to_list())

['Â£51.77', 'Â£53.74', 'Â£50.10', 'Â£47.82', 'Â£54.23', 'Â£22.65', 'Â£33.34', 'Â£17.93', 'Â£22.60', 'Â£52.15']


Clean Price and create price_gbp

In [52]:
# Clean the price and convert it to numeric GBP
df_clean["price_gbp"] = (
    df_clean["price"]
    .str.replace("Â£", "", regex=False)
    .str.replace("£", "", regex=False)
    .str.strip()
    .astype(float)
)

print(df_clean[["price", "price_gbp"]].head(10))
print("\nData type:", df_clean["price_gbp"].dtype)

     price  price_gbp
0  Â£51.77      51.77
1  Â£53.74      53.74
2  Â£50.10      50.10
3  Â£47.82      47.82
4  Â£54.23      54.23
5  Â£22.65      22.65
6  Â£33.34      33.34
7  Â£17.93      17.93
8  Â£22.60      22.60
9  Â£52.15      52.15

Data type: float64


Clean the Rating

In [53]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df_clean["rating"] = df_clean["rating"].map(rating_map)

print(df_clean[["title", "rating"]].head(10))
print("\nData type:", df_clean["rating"].dtype)

                                               title  rating
0                               A Light in the Attic       3
1                                 Tipping the Velvet       1
2                                         Soumission       1
3                                      Sharp Objects       4
4              Sapiens: A Brief History of Humankind       5
5                                    The Requiem Red       1
6  The Dirty Little Secrets of Getting Your Dream...       4
7  The Coming Woman: A Novel Based on the Life of...       3
8  The Boys in the Boat: Nine Americans and Their...       4
9                                    The Black Maria       1

Data type: int64


Next: Availability → in_stock

In [54]:
df_clean["in_stock"] = (
    df_clean["availability"]
    .str.contains("In stock", case=False, na=False)
)

print(df_clean[["availability", "in_stock"]].head(10))
print("\nData type:", df_clean["in_stock"].dtype)

  availability  in_stock
0     In stock      True
1     In stock      True
2     In stock      True
3     In stock      True
4     In stock      True
5     In stock      True
6     In stock      True
7     In stock      True
8     In stock      True
9     In stock      True

Data type: bool


Step 12 — Check data quality before GBP → INR

In [56]:
print("Missing values:")
print(df_clean.isnull().sum())

print("\nDuplicate rows:", df_clean.duplicated().sum())

Missing values:
title           0
price           0
rating          0
availability    0
category        0
price_gbp       0
in_stock        0
price_inr       0
dtype: int64

Duplicate rows: 0


Step 13 — Create price_inr

In [55]:
# Project-defined fixed conversion rate
GBP_TO_INR = 105.50

# Convert GBP price to INR
df_clean["price_inr"] = (
    df_clean["price_gbp"] * GBP_TO_INR
).round(2)

print(df_clean[["price_gbp", "price_inr"]].head(10))

print("\nConversion rate used:", GBP_TO_INR)
print("Data type:", df_clean["price_inr"].dtype)

   price_gbp  price_inr
0      51.77    5461.74
1      53.74    5669.57
2      50.10    5285.55
3      47.82    5045.01
4      54.23    5721.26
5      22.65    2389.57
6      33.34    3517.37
7      17.93    1891.62
8      22.60    2384.30
9      52.15    5501.82

Conversion rate used: 105.5
Data type: float64


In [20]:
print("Dataset shape:", df_clean.shape)

print("\nColumns:")
print(df_clean.columns.tolist())

print("\nData types:")
print(df_clean.dtypes)

print("\nMissing values:")
print(df_clean.isnull().sum())

Dataset shape: (80, 8)

Columns:
['title', 'price', 'rating', 'availability', 'category', 'price_gbp', 'in_stock', 'price_inr']

Data types:
title               str
price               str
rating            int64
availability        str
category            str
price_gbp       float64
in_stock           bool
price_inr       float64
dtype: object

Missing values:
title           0
price           0
rating          0
availability    0
category        0
price_gbp       0
in_stock        0
price_inr       0
dtype: int64


one final data-quality validation

In [21]:
print("Rating values:", sorted(df_clean["rating"].unique()))
print("Rating range valid:", df_clean["rating"].between(1, 5).all())

print("\nPrice GBP minimum:", df_clean["price_gbp"].min())
print("Price GBP valid:", (df_clean["price_gbp"] > 0).all())

print("\nPrice INR minimum:", df_clean["price_inr"].min())
print("Price INR valid:", (df_clean["price_inr"] > 0).all())

print("\nIn-stock data type:", df_clean["in_stock"].dtype)
print("Duplicate rows:", df_clean.duplicated().sum())

Rating values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Rating range valid: True

Price GBP minimum: 12.84
Price GBP valid: True

Price INR minimum: 1354.62
Price INR valid: True

In-stock data type: bool
Duplicate rows: 0


create the database

In [22]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

Current working directory:
c:\Users\DELL\Documents\zepto-capstone-project\data_pipeline\notebooks


Next step — create the SQLite database

In [23]:
import sqlite3
from pathlib import Path

# Create path to the data folder
data_folder = Path.cwd().parent / "data"
data_folder.mkdir(parents=True, exist_ok=True)

# SQLite database path
db_path = data_folder / "zepto_books.db"

print("Database path:")
print(db_path)

# Connect to SQLite
conn = sqlite3.connect(db_path)

# Enable foreign-key enforcement
conn.execute("PRAGMA foreign_keys = ON")

print("\nSQLite database created successfully!")

Database path:
c:\Users\DELL\Documents\zepto-capstone-project\data_pipeline\data\zepto_books.db

SQLite database created successfully!


Next step: create the two required tables

In [24]:
# Create the normalized database tables

create_categories_table = """
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE NOT NULL
);
"""

create_books_table = """
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
);
"""

conn.execute(create_categories_table)
conn.execute(create_books_table)
conn.commit()

print("Tables created successfully!")

Tables created successfully!


Next step: insert the categories

In [25]:
# Get unique categories from the cleaned dataset
categories = sorted(df_clean["category"].dropna().unique())

print("Number of unique categories:", len(categories))
print("\nFirst 10 categories:")
print(categories[:10])

# Insert categories into the categories table
conn.executemany(
    "INSERT OR IGNORE INTO categories (category_name) VALUES (?)",
    [(category,) for category in categories]
)

conn.commit()

print("\nCategories inserted successfully!")

Number of unique categories: 28

First 10 categories:
['Add a comment', 'Art', 'Business', 'Childrens', 'Contemporary', 'Default', 'Fantasy', 'Fiction', 'Food and Drink', 'Health']

Categories inserted successfully!


Next step: create the category ID mapping

In [26]:
# Read category IDs from SQLite
categories_df = pd.read_sql(
    "SELECT category_id, category_name FROM categories ORDER BY category_id",
    conn
)

print("Categories in SQLite:", len(categories_df))
print("\nCategory table:")
display(categories_df)

Categories in SQLite: 28

Category table:


,category_id,category_name
0,1,Add a comment
1,2,Art
2,3,Business
3,4,Childrens
4,5,Contemporary
5,6,Default
6,7,Fantasy
7,8,Fiction
8,9,Food and Drink
9,10,Health


Next step: map each book to category_id

In [27]:
# Create a category name -> category_id mapping

category_map = dict(
    zip(
        categories_df["category_name"],
        categories_df["category_id"]
    )
)

# Add category_id to the cleaned book data
books_for_db = df_clean.copy()

books_for_db["category_id"] = books_for_db["category"].map(category_map)

print("Total books:", len(books_for_db))
print("Missing category IDs:", books_for_db["category_id"].isnull().sum())

print("\nSample mapping:")
display(
    books_for_db[
        ["title", "category", "category_id"]
    ].head(10)
)

Total books: 80
Missing category IDs: 0

Sample mapping:


,title,category,category_id
0,A Light in the Attic,Poetry,19
1,Tipping the Velvet,Historical Fiction,11
2,Soumission,Fiction,8
3,Sharp Objects,Mystery,15
4,Sapiens: A Brief History of Humankind,History,12
5,The Requiem Red,Young Adult,28
6,The Dirty Little Secrets of Getting Your Dream...,Business,3
7,The Coming Woman: A Novel Based on the Life of...,Default,6
8,The Boys in the Boat: Nine Americans and Their...,Default,6
9,The Black Maria,Poetry,19


Next step: insert the 80 books

In [28]:
# Select only the columns required for the books table
books_to_insert = books_for_db[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_id"
    ]
].copy()

# Convert boolean to SQLite INTEGER (0/1)
books_to_insert["in_stock"] = books_to_insert["in_stock"].astype(int)

print("Rows ready for insertion:", len(books_to_insert))
print("\nColumns:")
print(books_to_insert.columns.tolist())

print("\nMissing values:")
print(books_to_insert.isnull().sum())

Rows ready for insertion: 80

Columns:
['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category_id']

Missing values:
title          0
price_gbp      0
price_inr      0
rating         0
in_stock       0
category_id    0
dtype: int64


In [57]:
# Defensive parsing checks required by the capstone

# Convert numeric fields safely; invalid values become NaN
df_clean["price_gbp"] = pd.to_numeric(
    df_clean["price_gbp"],
    errors="coerce"
)

df_clean["rating"] = pd.to_numeric(
    df_clean["rating"],
    errors="coerce"
)

# Check parsing failures
price_failures = df_clean["price_gbp"].isna().sum()
rating_failures = df_clean["rating"].isna().sum()

print("Price parsing failures:", price_failures)
print("Rating parsing failures:", rating_failures)

# Median-impute numeric parsing failures if any
if price_failures > 0:
    df_clean["price_gbp"] = df_clean["price_gbp"].fillna(
        df_clean["price_gbp"].median()
    )

if rating_failures > 0:
    df_clean["rating"] = df_clean["rating"].fillna(
        df_clean["rating"].median()
    ).round().astype(int)

# Recalculate INR from the cleaned GBP value
GBP_TO_INR = 105.50

df_clean["price_inr"] = (
    df_clean["price_gbp"] * GBP_TO_INR
).round(2)

print("\nFinal parsing check:")
print("Missing price_gbp:", df_clean["price_gbp"].isna().sum())
print("Missing rating:", df_clean["rating"].isna().sum())
print("Missing price_inr:", df_clean["price_inr"].isna().sum())

Price parsing failures: 0
Rating parsing failures: 0

Final parsing check:
Missing price_gbp: 0
Missing rating: 0
Missing price_inr: 0


Next cell — insert the 80 books

In [30]:
# Insert cleaned books into SQLite

insert_books_sql = """
INSERT INTO books (
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock,
    category_id
)
VALUES (?, ?, ?, ?, ?, ?)
"""

book_records = list(
    books_to_insert.itertuples(
        index=False,
        name=None
    )
)

conn.executemany(insert_books_sql, book_records)
conn.commit()

print("Books inserted successfully!")
print("Rows inserted:", len(book_records))

Books inserted successfully!
Rows inserted: 80


Next step — verify the actual SQLite database

In [31]:
# Verify actual row counts in SQLite

category_count = pd.read_sql(
    "SELECT COUNT(*) AS category_count FROM categories",
    conn
)

book_count = pd.read_sql(
    "SELECT COUNT(*) AS book_count FROM books",
    conn
)

print("Categories:")
display(category_count)

print("\nBooks:")
display(book_count)

Categories:


,category_count
0,28



Books:


,book_count
0,80


Next step — verify PK/FK integrity

In [32]:
# Verify SQLite foreign-key integrity

foreign_key_check = pd.read_sql(
    "PRAGMA foreign_key_check;",
    conn
)

print("Foreign-key violations found:", len(foreign_key_check))

if len(foreign_key_check) == 0:
    print("Foreign-key integrity check: PASS")
else:
    display(foreign_key_check)

Foreign-key violations found: 0
Foreign-key integrity check: PASS


Next: SQL Query 1

In [60]:
# SQL Query 1
# Requirement covered: SELECT + WHERE + ORDER BY + LIMIT

query_1 = """
SELECT
    title,
    price_gbp,
    price_inr,
    rating
FROM books
WHERE price_gbp > 40
ORDER BY price_gbp DESC
LIMIT 10;
"""

query_1_result = pd.read_sql(query_1, conn)

print("Query 1: Books priced above £40, highest price first")
display(query_1_result)

Query 1: Books priced above £40, highest price first


,title,price_gbp,price_inr,rating
0,The Death of Humanity: and the Case for Life,58.11,6130.60,4
1,Slow States of Collapse: Poems,57.31,6046.20,3
2,Our Band Could Be Your Life: Scenes from the A...,57.25,6039.88,3
3,The Past Never Ends,56.50,5960.75,4
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,5951.25,1
5,Masks and Shadows,56.40,5950.20,2
6,The Secret of Dreadwillow Carse,56.13,5921.72,1
7,The Electric Pencil: Drawings from Inside Stat...,56.06,5914.33,1
8,Birdsong: A Story in Pictures,54.64,5764.52,3
9,Sapiens: A Brief History of Humankind,54.23,5721.26,5


Query 2 — DISTINCT

In [61]:
# SQL Query 2
# Requirement covered: DISTINCT

query_2 = """
SELECT DISTINCT category_name
FROM categories
ORDER BY category_name;
"""

query_2_result = pd.read_sql(query_2, conn)

print("Query 2: Distinct book categories")
display(query_2_result)

print("\nNumber of distinct categories:", len(query_2_result))

Query 2: Distinct book categories


,category_name
0,Add a comment
1,Art
2,Business
3,Childrens
4,Contemporary
5,Default
6,Fantasy
7,Fiction
8,Food and Drink
9,Health



Number of distinct categories: 29


Query 3 — BETWEEN

In [62]:
# SQL Query 3
# Requirement covered: BETWEEN

query_3 = """
SELECT
    title,
    price_gbp,
    price_inr,
    rating
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp ASC;
"""

query_3_result = pd.read_sql(query_3, conn)

print("Query 3: Books priced between £20 and £40")
display(query_3_result)

print("\nNumber of books found:", len(query_3_result))

Query 3: Books priced between £20 and £40


,title,price_gbp,price_inr,rating
0,The Inefficiency Assassin: Time Management Tac...,20.59,2172.24,5
1,Shakespeare's Sonnets,20.66,2179.63,4
2,In the Country We Love: My Family Divided,22.00,2321.00,4
3,America's Cradle of Quarterbacks: Western Penn...,22.50,2373.75,3
4,The Boys in the Boat: Nine Americans and Their...,22.60,2384.30,4
5,The Requiem Red,22.65,2389.57,1
6,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,2438.10,5
7,The Elephant Tree,23.82,2513.01,5
8,Olio,23.88,2519.34,1
9,The Mindfulness and Acceptance Workbook for An...,23.89,2520.40,4



Number of books found: 34


Next: Query 4 — JOIN

In [63]:
# SQL Query 4
# Requirement covered: JOIN

query_4 = """
SELECT
    b.title,
    c.category_name,
    b.price_gbp,
    b.price_inr,
    b.rating
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10;
"""

query_4_result = pd.read_sql(query_4, conn)

print("Query 4: Top-rated books with their categories")
display(query_4_result)

print("\nNumber of books returned:", len(query_4_result))

Query 4: Top-rated books with their categories


,title,category_name,price_gbp,price_inr,rating
0,Sapiens: A Brief History of Humankind,History,54.23,5721.26,5
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29,5516.60,5
2,"We Love You, Charlie Freeman",Fiction,50.27,5303.48,5
3,Private Paris (Private #10),Fiction,47.61,5022.85,5
4,Worlds Elsewhere: Journeys Around Shakespeareâ...,Nonfiction,40.30,4251.65,5
5,Join,Science Fiction,35.67,3763.19,5
6,Rip it Up and Start Again,Music,35.02,3694.61,5
7,Black Dust,Romance,34.53,3642.92,5
8,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,32.24,3401.32,5
9,Chase Me (Paris Nights #2),Romance,25.27,2665.98,5



Number of books returned: 10


Query 5 — IN

In [64]:
# SQL Query 5
# Requirement covered: IN

query_5 = """
SELECT
    b.title,
    c.category_name,
    b.price_gbp,
    b.rating
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
WHERE c.category_name IN ('Poetry', 'Mystery', 'History')
ORDER BY b.price_gbp DESC;
"""

query_5_result = pd.read_sql(query_5, conn)

print("Query 5: Books from Poetry, Mystery, or History")
display(query_5_result)

print("\nNumber of books returned:", len(query_5_result))

Query 5: Books from Poetry, Mystery, or History


,title,category_name,price_gbp,rating
0,Slow States of Collapse: Poems,Poetry,57.31,3
1,The Past Never Ends,Mystery,56.50,4
2,Sapiens: A Brief History of Humankind,History,54.23,5
3,The Black Maria,Poetry,52.15,1
4,A Light in the Attic,Poetry,51.77,3
5,Sharp Objects,Mystery,47.82,4
6,"Political Suicide: Missteps, Peccadilloes, Bad...",History,36.28,2
7,You can't bury them all: Poems,Poetry,33.63,2
8,"Unbound: How Eight Technologies Made Us Human,...",History,25.52,1
9,Olio,Poetry,23.88,1



Number of books returned: 14


Next step: save all 5 SQL queries

In [38]:
# Verify all 5 SQL queries are available

queries = {
    "Query 1": query_1,
    "Query 2": query_2,
    "Query 3": query_3,
    "Query 4": query_4,
    "Query 5": query_5
}

for name, query in queries.items():
    print(f"\n{name}:")
    print(query.strip())


Query 1:
SELECT
    title,
    price_gbp,
    price_inr,
    rating
FROM books
WHERE price_gbp > 40
ORDER BY price_gbp DESC
LIMIT 10;

Query 2:
SELECT DISTINCT category_name
FROM categories
ORDER BY category_name;

Query 3:
SELECT
    title,
    price_gbp,
    price_inr,
    rating
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp ASC;

Query 4:
SELECT
    b.title,
    c.category_name,
    b.price_gbp,
    b.price_inr,
    b.rating
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10;

Query 5:
SELECT
    b.title,
    c.category_name,
    b.price_gbp,
    b.rating
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
WHERE c.category_name IN ('Poetry', 'Mystery', 'History')
ORDER BY b.price_gbp DESC;


Let's verify all 5 cleanly

In [39]:
# Verify that all 5 SQL queries exist

print("Query 1:", "Available" if query_1 else "Missing")
print("Query 2:", "Available" if query_2 else "Missing")
print("Query 3:", "Available" if query_3 else "Missing")
print("Query 4:", "Available" if query_4 else "Missing")
print("Query 5:", "Available" if query_5 else "Missing")

Query 1: Available
Query 2: Available
Query 3: Available
Query 4: Available
Query 5: Available


Step 3B — Generate the outputs automatically

In [65]:
from pathlib import Path

sql_folder = Path.cwd().parent / "sql"
output_file = sql_folder / "query_outputs.md"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("# SQL Query Outputs\n\n")

    results = {
        "Query 1": query_1_result,
        "Query 2": query_2_result,
        "Query 3": query_3_result,
        "Query 4": query_4_result,
        "Query 5": query_5_result
    }

    for name, result in results.items():
        f.write(f"## {name}\n\n")
        f.write("```text\n")
        f.write(result.to_string(index=False))
        f.write("\n```\n\n")

print(f"Saved query outputs to: {output_file}")

Saved query outputs to: c:\Users\DELL\Documents\zepto-capstone-project\data_pipeline\sql\query_outputs.md


Create the pd.merge() result

In [69]:
# Reproduce the SQL JOIN using pandas merge

books_for_merge = books_for_db[
    ["title", "price_gbp", "price_inr", "rating", "category_id"]
].copy()

merge_join_result = books_for_merge.merge(
    categories_df,
    on="category_id",
    how="inner"
)

merge_join_result = merge_join_result[
    ["title", "category_name", "price_gbp", "price_inr", "rating"]
]

merge_join_result = (
    merge_join_result
    .sort_values(
        ["rating", "price_gbp"],
        ascending=[False, False]
    )
    .head(10)
    .reset_index(drop=True)
)

print("Pandas merge JOIN result:")
display(merge_join_result)

Pandas merge JOIN result:


,title,category_name,price_gbp,price_inr,rating
0,Sapiens: A Brief History of Humankind,History,54.23,5721.26,5
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29,5516.60,5
2,"We Love You, Charlie Freeman",Fiction,50.27,5303.48,5
3,Private Paris (Private #10),Fiction,47.61,5022.85,5
4,Worlds Elsewhere: Journeys Around Shakespeareâ...,Nonfiction,40.30,4251.65,5
5,Join,Science Fiction,35.67,3763.19,5
6,Rip it Up and Start Again,Music,35.02,3694.61,5
7,Black Dust,Romance,34.53,3642.92,5
8,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,32.24,3401.32,5
9,Chase Me (Paris Nights #2),Romance,25.27,2665.98,5


Now compare SQL JOIN vs Pandas JOIN

In [70]:
# Compare SQL JOIN result with pandas merge result

sql_join_result = (
    query_4_result[
        ["title", "category_name", "price_gbp", "price_inr", "rating"]
    ]
    .sort_values(
        ["rating", "price_gbp"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

pandas_join_result = (
    merge_join_result
    .sort_values(
        ["rating", "price_gbp"],
        ascending=[False, False]
    )
    .head(10)
    .reset_index(drop=True)
)

print("SQL JOIN rows:", len(sql_join_result))
print("Pandas merge rows:", len(pandas_join_result))

print(
    "SQL JOIN and pandas merge match:",
    sql_join_result.equals(pandas_join_result)
)

SQL JOIN rows: 10
Pandas merge rows: 10
SQL JOIN and pandas merge match: True


In [67]:
print("SQL JOIN columns:")
print(sql_join_result.columns.tolist())

print("\nPandas merge columns:")
print(pandas_join_result.columns.tolist())

print("\nSQL JOIN dtypes:")
print(sql_join_result.dtypes)

print("\nPandas merge dtypes:")
print(pandas_join_result.dtypes)

print("\nSQL JOIN result:")
display(sql_join_result)

print("\nPandas merge result:")
display(pandas_join_result)

SQL JOIN columns:
['title', 'category_name', 'price_gbp', 'price_inr', 'rating']

Pandas merge columns:
['title', 'category_name', 'price_gbp', 'price_inr', 'rating']

SQL JOIN dtypes:
title                str
category_name        str
price_gbp        float64
price_inr        float64
rating             int64
dtype: object

Pandas merge dtypes:
title                str
category_name        str
price_gbp        float64
price_inr        float64
rating             int64
dtype: object

SQL JOIN result:


,title,category_name,price_gbp,price_inr,rating
0,Sapiens: A Brief History of Humankind,History,54.23,5721.26,5
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29,5516.60,5
2,"We Love You, Charlie Freeman",Fiction,50.27,5303.48,5
3,Private Paris (Private #10),Fiction,47.61,5022.85,5
4,Worlds Elsewhere: Journeys Around Shakespeareâ...,Nonfiction,40.30,4251.65,5
5,Join,Science Fiction,35.67,3763.19,5
6,Rip it Up and Start Again,Music,35.02,3694.61,5
7,Black Dust,Romance,34.53,3642.92,5
8,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,32.24,3401.32,5
9,Chase Me (Paris Nights #2),Romance,25.27,2665.98,5



Pandas merge result:


,title,category_name,price_gbp,price_inr,rating
0,Sapiens: A Brief History of Humankind,History,54.23,5721.26,5
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29,5516.60,5
2,"We Love You, Charlie Freeman",Fiction,50.27,5303.48,5
3,Private Paris (Private #10),Fiction,47.61,5022.85,5
4,Worlds Elsewhere: Journeys Around Shakespeareâ...,Nonfiction,40.30,4251.65,5
5,Rip it Up and Start Again,Music,35.02,3694.61,5
6,Black Dust,Romance,34.53,3642.92,5
7,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,32.24,3401.32,5
8,Chase Me (Paris Nights #2),Romance,25.27,2665.98,5
9,The Elephant Tree,Thriller,23.82,2513.01,5


Add the final validation cell

In [43]:
# Module 1 - Final Validation Checklist

from pathlib import Path
import pandas as pd

print("========== MODULE 1 FINAL VALIDATION ==========\n")

# 1. Dataset size
print("1. Dataset size")
print("   Books:", len(df_clean))
print("   Categories:", df_clean["category"].nunique())
print("   PASS:", len(df_clean) >= 60 and df_clean["category"].nunique() >= 3)

# 2. Required columns
required_columns = [
    "title",
    "price_gbp",
    "price_inr",
    "rating",
    "in_stock",
    "category"
]

print("\n2. Required columns")
print("   PASS:", all(col in df_clean.columns for col in required_columns))

# 3. Data types
print("\n3. Data types")
print("   price_gbp numeric:", pd.api.types.is_numeric_dtype(df_clean["price_gbp"]))
print("   price_inr numeric:", pd.api.types.is_numeric_dtype(df_clean["price_inr"]))
print("   rating integer:", pd.api.types.is_integer_dtype(df_clean["rating"]))
print("   in_stock boolean:", pd.api.types.is_bool_dtype(df_clean["in_stock"]))

# 4. Rating validation
print("\n4. Rating validation")
valid_ratings = set(df_clean["rating"].dropna().unique()).issubset({1, 2, 3, 4, 5})
print("   Valid ratings 1-5:", valid_ratings)

# 5. Missing values
print("\n5. Missing values")
missing_values = df_clean[
    ["title", "price_gbp", "price_inr", "rating", "in_stock", "category"]
].isna().sum()

print(missing_values)
print("   PASS:", missing_values.sum() == 0)

# 6. GBP to INR conversion
print("\n6. GBP to INR conversion")
GBP_TO_INR = 105.50
conversion_check = (
    (df_clean["price_inr"] - (df_clean["price_gbp"] * GBP_TO_INR).round(2))
    .abs()
    .max()
)

print("   Required rate: 1 GBP =", GBP_TO_INR, "INR")
print("   Maximum conversion difference:", conversion_check)
print("   PASS:", conversion_check == 0)

# 7. Duplicate rows
print("\n7. Duplicate rows")
print("   Duplicate rows:", df_clean.duplicated().sum())
print("   PASS:", df_clean.duplicated().sum() == 0)

# 8. SQLite validation
print("\n8. SQLite validation")

books_count = pd.read_sql(
    "SELECT COUNT(*) AS count FROM books",
    conn
).iloc[0]["count"]

categories_count = pd.read_sql(
    "SELECT COUNT(*) AS count FROM categories",
    conn
).iloc[0]["count"]

print("   Books in SQLite:", books_count)
print("   Categories in SQLite:", categories_count)

foreign_key_check = pd.read_sql(
    "PRAGMA foreign_key_check;",
    conn
)

print("   Foreign key violations:", len(foreign_key_check))
print(
    "   PASS:",
    books_count == len(df_clean)
    and categories_count == df_clean["category"].nunique()
    and len(foreign_key_check) == 0
)

# 9. SQL queries and outputs
print("\n9. SQL query files")

sql_folder = Path.cwd().parent / "sql"
queries_file = sql_folder / "queries.sql"
outputs_file = sql_folder / "query_outputs.md"

print("   queries.sql exists:", queries_file.exists())
print("   query_outputs.md exists:", outputs_file.exists())
print("   Five query variables available:",
      all([
          bool(query_1),
          bool(query_2),
          bool(query_3),
          bool(query_4),
          bool(query_5)
      ]))

# 10. SQL JOIN vs pandas merge
print("\n10. SQL JOIN vs pandas merge")
print("   SQL JOIN rows:", len(sql_join_result))
print("   Pandas merge rows:", len(pandas_join_result))
print("   Results match:", sql_join_result.equals(pandas_join_result))

print("\n========== FINAL STATUS ==========")
print("MODULE 1 VALIDATION COMPLETE")

========== MODULE 1 FINAL VALIDATION ==========

1. Dataset size
   Books: 80
   Categories: 28
   PASS: True

2. Required columns
   PASS: True

3. Data types
   price_gbp numeric: True
   price_inr numeric: True
   rating integer: True
   in_stock boolean: True

4. Rating validation
   Valid ratings 1-5: True

5. Missing values
title        0
price_gbp    0
price_inr    0
rating       0
in_stock     0
category     0
dtype: int64
   PASS: True

6. GBP to INR conversion
   Required rate: 1 GBP = 105.5 INR
   Maximum conversion difference: 0.0
   PASS: True

7. Duplicate rows
   Duplicate rows: 0
   PASS: True

8. SQLite validation
   Books in SQLite: 80
   Categories in SQLite: 28
   Foreign key violations: 0
   PASS: True

9. SQL query files
   queries.sql exists: True
   query_outputs.md exists: True
   Five query variables available: True

10. SQL JOIN vs pandas merge
   SQL JOIN rows: 10
   Pandas merge rows: 10
   Results match: True

========== FINAL STATUS ==========
MODULE 1 VA

Rebuild SQLite database from the final 100-book dataset

In [58]:
# Rebuild SQLite database from the final 100-book dataset

import sqlite3
from pathlib import Path

# Close the old connection
try:
    conn.close()
except:
    pass

# Database path
data_folder = Path.cwd().parent / "data"
data_folder.mkdir(parents=True, exist_ok=True)
db_path = data_folder / "zepto_books.db"

# Remove old database so we rebuild it cleanly
if db_path.exists():
    db_path.unlink()

# Create fresh database
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON")

# Create normalized tables
conn.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE NOT NULL
)
""")

conn.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

# Insert categories
unique_categories = sorted(df_clean["category"].unique())

conn.executemany(
    "INSERT INTO categories (category_name) VALUES (?)",
    [(category,) for category in unique_categories]
)

# Read category IDs
categories_df = pd.read_sql(
    "SELECT category_id, category_name FROM categories ORDER BY category_id",
    conn
)

# Map category names to IDs
category_map = dict(
    zip(
        categories_df["category_name"],
        categories_df["category_id"]
    )
)

# Prepare books for SQLite
books_for_db = df_clean.copy()
books_for_db["category_id"] = books_for_db["category"].map(category_map)

books_to_insert = books_for_db[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_id"
    ]
].copy()

# SQLite stores boolean values as INTEGER 0/1
books_to_insert["in_stock"] = books_to_insert["in_stock"].astype(int)

# Insert books
insert_books_sql = """
INSERT INTO books (
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock,
    category_id
)
VALUES (?, ?, ?, ?, ?, ?)
"""

book_records = list(
    books_to_insert.itertuples(index=False, name=None)
)

conn.executemany(insert_books_sql, book_records)
conn.commit()

# Verify database
books_count = pd.read_sql(
    "SELECT COUNT(*) AS count FROM books",
    conn
).iloc[0]["count"]

categories_count = pd.read_sql(
    "SELECT COUNT(*) AS count FROM categories",
    conn
).iloc[0]["count"]

foreign_key_check = pd.read_sql(
    "PRAGMA foreign_key_check;",
    conn
)

print("SQLite database rebuilt successfully.")
print("Books inserted:", books_count)
print("Categories inserted:", categories_count)
print("Foreign key violations:", len(foreign_key_check))
print("Database path:", db_path)

SQLite database rebuilt successfully.
Books inserted: 100
Categories inserted: 29
Foreign key violations: 0
Database path: c:\Users\DELL\Documents\zepto-capstone-project\data_pipeline\data\zepto_books.db


In [59]:
# Verify rebuilt SQLite database

books_check = pd.read_sql(
    "SELECT COUNT(*) AS book_count FROM books",
    conn
)

categories_check = pd.read_sql(
    "SELECT COUNT(*) AS category_count FROM categories",
    conn
)

print("Books in SQLite:", books_check.iloc[0]["book_count"])
print("Categories in SQLite:", categories_check.iloc[0]["category_count"])

print(
    "Book count matches df_clean:",
    books_check.iloc[0]["book_count"] == len(df_clean)
)

print(
    "Category count matches df_clean:",
    categories_check.iloc[0]["category_count"]
    == df_clean["category"].nunique()
)

Books in SQLite: 100
Categories in SQLite: 29
Book count matches df_clean: True
Category count matches df_clean: True


Final Validation with 100 books

In [71]:
# ==========================================
# FINAL MODULE 1 ACCEPTANCE VALIDATION
# ==========================================

from pathlib import Path

print("===== MODULE 1 FINAL VALIDATION =====")

# 1. Dataset size
book_count = len(df_clean)
category_count = df_clean["category"].nunique()

print("\n1. Dataset size")
print("Books:", book_count)
print("Categories:", category_count)
print("Minimum book requirement met:", book_count >= 60)
print("Minimum category requirement met:", category_count >= 3)

# 2. Required columns
required_columns = [
    "title",
    "price_gbp",
    "rating",
    "in_stock",
    "category",
    "price_inr"
]

missing_columns = [
    col for col in required_columns
    if col not in df_clean.columns
]

print("\n2. Required columns")
print("Missing columns:", missing_columns)
print("All required columns present:", len(missing_columns) == 0)

# 3. Data type and value validation
print("\n3. Data quality and types")

print("price_gbp numeric:", df_clean["price_gbp"].dtype.kind in "fi")
print("rating integer:", df_clean["rating"].dtype.kind in "iu")
print("rating values valid:", df_clean["rating"].between(1, 5).all())
print("in_stock boolean:", df_clean["in_stock"].dtype == bool)
print("price_inr numeric:", df_clean["price_inr"].dtype.kind in "fi")

print("Missing price_gbp:", df_clean["price_gbp"].isna().sum())
print("Missing rating:", df_clean["rating"].isna().sum())
print("Missing price_inr:", df_clean["price_inr"].isna().sum())

# 4. Currency conversion validation
GBP_TO_INR = 105.50

conversion_check = (
    df_clean["price_inr"]
    .eq((df_clean["price_gbp"] * GBP_TO_INR).round(2))
    .all()
)

print("\n4. Currency conversion")
print("Required rate: 1 GBP = 105.50 INR")
print("Conversion correct:", conversion_check)

# 5. Duplicate validation
print("\n5. Duplicate validation")
print("Duplicate rows:", df_clean.duplicated().sum())

# 6. SQLite validation
print("\n6. SQLite database")

print("Database exists:", Path(db_path).exists())

sqlite_books = pd.read_sql(
    "SELECT COUNT(*) AS count FROM books;",
    conn
)["count"].iloc[0]

sqlite_categories = pd.read_sql(
    "SELECT COUNT(*) AS count FROM categories;",
    conn
)["count"].iloc[0]

print("Books in SQLite:", sqlite_books)
print("Categories in SQLite:", sqlite_categories)
print("SQLite book count matches:", sqlite_books == book_count)
print("SQLite category count matches:", sqlite_categories == category_count)

# 7. SQL query validation
print("\n7. SQL queries")

queries_available = all([
    query_1,
    query_2,
    query_3,
    query_4,
    query_5
])

print("5 required queries available:", queries_available)

print("Query 1 rows:", len(query_1_result))
print("Query 2 rows:", len(query_2_result))
print("Query 3 rows:", len(query_3_result))
print("Query 4 rows:", len(query_4_result))
print("Query 5 rows:", len(query_5_result))

# 8. pd.merge JOIN validation
print("\n8. SQL JOIN vs pandas merge")

print("SQL JOIN rows:", len(sql_join_result))
print("Pandas merge rows:", len(pandas_join_result))
print("JOIN results match:", sql_join_result.equals(pandas_join_result))

# 9. Final overall check
all_checks_pass = all([
    book_count >= 60,
    category_count >= 3,
    len(missing_columns) == 0,
    df_clean["price_gbp"].dtype.kind in "fi",
    df_clean["rating"].dtype.kind in "iu",
    df_clean["rating"].between(1, 5).all(),
    df_clean["in_stock"].dtype == bool,
    df_clean["price_inr"].dtype.kind in "fi",
    df_clean["price_gbp"].isna().sum() == 0,
    df_clean["rating"].isna().sum() == 0,
    df_clean["price_inr"].isna().sum() == 0,
    conversion_check,
    df_clean.duplicated().sum() == 0,
    Path(db_path).exists(),
    sqlite_books == book_count,
    sqlite_categories == category_count,
    queries_available,
    sql_join_result.equals(pandas_join_result)
])

print("\n====================================")
print("FINAL MODULE 1 STATUS:", "PASS" if all_checks_pass else "CHECK REQUIRED")
print("====================================")

===== MODULE 1 FINAL VALIDATION =====

1. Dataset size
Books: 100
Categories: 29
Minimum book requirement met: True
Minimum category requirement met: True

2. Required columns
Missing columns: []
All required columns present: True

3. Data quality and types
price_gbp numeric: True
rating integer: True
rating values valid: True
in_stock boolean: True
price_inr numeric: True
Missing price_gbp: 0
Missing rating: 0
Missing price_inr: 0

4. Currency conversion
Required rate: 1 GBP = 105.50 INR
Conversion correct: True

5. Duplicate validation
Duplicate rows: 0

6. SQLite database
Database exists: True
Books in SQLite: 100
Categories in SQLite: 29
SQLite book count matches: True
SQLite category count matches: True

7. SQL queries
5 required queries available: True
Query 1 rows: 10
Query 2 rows: 29
Query 3 rows: 34
Query 4 rows: 10
Query 5 rows: 14

8. SQL JOIN vs pandas merge
SQL JOIN rows: 10
Pandas merge rows: 10
JOIN results match: True

FINAL MODULE 1 STATUS: PASS
